# Llama 3.3 70B Optimized Notebook

Optimized version of the 3-column notebook for `meta-llama/Llama-3.3-70B-Instruct`.

Optimization goals:
- lower memory pressure
- more stable 2-GPU execution
- chunked inference instead of one massive generation call
- shorter outputs and cleaner parsing


## 1) Imports

In [ ]:
import os
import gc
import json
import re
import subprocess
import time
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import HTML, display
from vllm import LLM, SamplingParams


## 2) Runtime configuration

In [ ]:
CUDA_VISIBLE_DEVICES = "0,1"
TENSOR_PARALLEL_SIZE = 2

os.environ["CUDA_VISIBLE_DEVICES"] = CUDA_VISIBLE_DEVICES
os.environ["VLLM_ENABLE_V1_MULTIPROCESSING"] = "1"
os.environ["MKL_THREADING_LAYER"] = "GNU"
os.environ["MKL_SERVICE_FORCE_INTEL"] = "1"

print("CUDA_VISIBLE_DEVICES:", os.environ.get("CUDA_VISIBLE_DEVICES"))
print("TENSOR_PARALLEL_SIZE:", TENSOR_PARALLEL_SIZE)


## 3) Paths

In [ ]:
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "annotation_working_master_human_2100_seed.csv").exists():
    PROJECT_ROOT = Path("/Users/raresolteanu/Desktop/Gliner-Work.Dauphine")

DATASET_PATH = PROJECT_ROOT / "annotation_working_master_human_2100_seed.csv"
OUTPUT_ROOT = PROJECT_ROOT / "communication_function_outputs" / "llama33_70b_3col_optimized"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
LATEX_DIR = PROJECT_ROOT / "latex" / "llama33_70b_3col_optimized"
LATEX_DIR.mkdir(parents=True, exist_ok=True)


## 4) Model configuration

In [ ]:
MODEL_NAME = "meta-llama/Llama-3.3-70B-Instruct"
CHAT_MODE = "hf_auto"
MODEL_SLUG = "meta_llama_llama_3_3_70b_instruct"

TEXT_COLUMNS = ["Script", "Titre", "Visuel"]

# More conservative generation settings than the earlier notebook.
TEMPERATURE = 0.0
MAX_NEW_TOKENS = 180
MAX_MODEL_LEN = 6144
VLLM_GPU_MEMORY_UTILIZATION = 0.88
VLLM_SWAP_SPACE_GB = 12
MODEL_DTYPE = "bfloat16"

# Execution tuning
SMOKE_TEST_N = 20
FULL_RUN_N = None
PROMPT_BATCH_SIZE = 128

MODEL_KWARGS: dict[str, Any] = {
    "tensor_parallel_size": TENSOR_PARALLEL_SIZE,
    "trust_remote_code": True,
    "gpu_memory_utilization": VLLM_GPU_MEMORY_UTILIZATION,
    "dtype": MODEL_DTYPE,
    "max_model_len": int(MAX_MODEL_LEN),
    "swap_space": int(VLLM_SWAP_SPACE_GB),
}

print("MODEL_NAME:", MODEL_NAME)
print("PROMPT_BATCH_SIZE:", PROMPT_BATCH_SIZE)
print("MAX_NEW_TOKENS:", MAX_NEW_TOKENS)
print("MAX_MODEL_LEN:", MAX_MODEL_LEN)


## 5) Prompt

In [ ]:
RUBRIC_TEXT = 'You are an expert annotation assistant for French automotive advertising.\n\nYour job is to analyze one ad and score it on three communication dimensions:\n- informativeness\n- expressiveness\n- phatic\n\nYou must annotate carefully and conservatively.\nYour goal is not to be creative.\nYour goal is to produce the most defensible annotation possible from the evidence in the ad.\n\nYou will receive ad text that may combine:\n- script\n- on-screen text\n- title\n- visual description\n\nThese elements may contain both literal information and symbolic or rhetorical cues.\nYou must judge the ad as a whole.\n\n==================================================\nTASK\n==================================================\n\nScore the ad on each dimension from 1 to 5.\n\n1 = almost absent\n2 = weak\n3 = moderate\n4 = strong\n5 = very strong\n\nThe three dimensions are independent.\nAn ad can be high on more than one dimension.\nDo not force the scores to sum to any fixed total.\n\nAfter scoring, choose:\n- dominant_dimension\n- dominant_dimension_score\n- confidence\n- reason\n\nIf the highest score is shared by more than one dimension, dominant_dimension must be "mixed".\n\nReturn strict JSON only.\n\n==================================================\nDIMENSION DEFINITIONS\n==================================================\n\nA. INFORMATIVENESS\n\nDefinition:\nHow much the ad provides factual, concrete, product-related, offer-related, or technically useful information.\n\nThis includes:\n- vehicle specifications\n- features and equipment\n- engine or powertrain information\n- electric or hybrid technology\n- charging, range, battery, consumption\n- safety systems\n- comfort or space features when presented concretely\n- maintenance, guarantee, reliability claims when concrete\n- price\n- discounts\n- financing, leasing, monthly payments\n- trade-in conditions\n- bonus or subsidy information\n- model names, versions, product details\n- explicit comparative or functional claims\n\nB. EXPRESSIVENESS\n\nDefinition:\nHow much the ad relies on emotion, desire, identity, aspiration, style, symbolic value, atmosphere, prestige, seduction, or aesthetic projection.\n\nC. PHATIC\n\nDefinition:\nHow much the ad creates, maintains, or foregrounds social contact, relational connection, conversational closeness, complicity, or audience bonding.\n\n==================================================\nKEY DISTINCTIONS\n==================================================\n\n- Informativeness = concrete useful product/offer content\n- Expressiveness = emotional or aesthetic persuasion\n- Phatic = contact, social bond, conversational connection\n\nHumor alone is not automatically phatic.\nEmotion alone is not automatically phatic.\nPrice or financing details increase informativeness.\n\n==================================================\nVERY IMPORTANT SCORING RULES\n==================================================\n\nUse the full scale carefully.\nDo not inflate all dimensions.\nDo not reward every polished ad with high expressiveness.\nDo not reward every second-person phrase with high phatic.\nDo not ignore concrete offer or product information.\n\nIf the ad combines a strong emotional frame with many concrete offer details, it may be high in both informativeness and expressiveness.\n\n==================================================\nSPECIAL AUTOMOTIVE RULES\n==================================================\n\nIn French car ads, the following usually increase informativeness:\n- monthly payment\n- leasing conditions\n- trade-in offers\n- ecological bonus\n- hybrid / electric / rechargeable wording\n- battery / charging / autonomy / range\n- horsepower, engine, consumption\n- guarantee, maintenance, equipment\n- product version, trim, pack, included options\n\n==================================================\nDECISION PROCEDURE\n==================================================\n\nStep 1. Identify the ad’s primary communicative force.\nStep 2. Identify the strongest evidence for each dimension.\nStep 3. Assign the three scores independently.\nStep 4. Determine the dominant dimension from the highest score.\nStep 5. Set dominant_dimension_score equal to the highest score.\nStep 6. Give a very short reason based only on actual evidence from the ad.\n\n==================================================\nOUTPUT REQUIREMENTS\n==================================================\n\nReturn strict JSON only.\n\n{\n  "informativeness": 1,\n  "expressiveness": 1,\n  "phatic": 1,\n  "dominant_dimension": "informativeness|expressiveness|phatic|mixed",\n  "dominant_dimension_score": 1,\n  "confidence": 0.0,\n  "reason": "short explanation"\n}\n\n{AD_TEXT}\n\nReturn only strict JSON.'

def output_schema_text() -> str:
    return '''{
  "informativeness": 1,
  "expressiveness": 1,
  "phatic": 1,
  "dominant_dimension": "informativeness|expressiveness|phatic|mixed",
  "dominant_dimension_score": 1,
  "confidence": 0.0,
  "reason": "short explanation"
}'''


## 6) Helpers

In [ ]:
def html_box(title: str, body: str, color: str = "#1f4e79", bg: str = "#eef6fb"):
    display(HTML(
        f"""
        <div style="border-left: 6px solid {color}; background:{bg}; padding:10px 14px; margin:8px 0; border-radius:6px;">
            <div style="font-weight:700; margin-bottom:4px;">{title}</div>
            <div style="white-space:pre-wrap;">{body}</div>
        </div>
        """
    ))

def clean_text(value) -> str:
    text = "" if pd.isna(value) else str(value)
    text = text.replace("\r", " ").replace("\n", " ").replace("\t", " ")
    text = re.sub(r"[\u200b\u200c\u200d\ufeff]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def join_labeled_parts(parts: list[tuple[str, str]]) -> str:
    kept = []
    for label, text in parts:
        text = clean_text(text)
        if text:
            kept.append(f"{label}: {text}")
    return "\n".join(kept).strip()

def build_input_text(row: pd.Series) -> str:
    parts = [(col, row.get(col, "")) for col in TEXT_COLUMNS]
    return join_labeled_parts(parts)

def build_prompt_content(ad_text: str) -> str:
    return RUBRIC_TEXT.replace("{AD_TEXT}", " ".join(str(ad_text).split()).strip())

def render_prompt_for_model(content: str, llm=None) -> str:
    if CHAT_MODE == "plain" or llm is None:
        return content
    tokenizer = llm.get_tokenizer()
    messages = [{"role": "user", "content": content}]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

def build_prompt(ad_text: str, llm=None) -> str:
    return render_prompt_for_model(build_prompt_content(ad_text), llm=llm)

def clamp_score(value, default=1):
    try:
        score = int(round(float(value)))
    except Exception:
        score = default
    return max(1, min(5, score))

def normalize_dimension_name(value):
    text = str(value or "").strip().lower()
    aliases = {
        "informative": "informativeness",
        "information": "informativeness",
        "referential": "informativeness",
        "expressive": "expressiveness",
        "emotive": "expressiveness",
        "emotion": "expressiveness",
        "phatique": "phatic",
    }
    text = aliases.get(text, text)
    return text if text in ["informativeness", "expressiveness", "phatic", "mixed"] else ""

def extract_json_object(text: str) -> dict:
    fenced = re.search(r"```(?:json)?\s*(\{.*?\})\s*```", text, flags=re.DOTALL)
    candidates = [fenced.group(1)] if fenced else []
    candidates.append(text)
    decoder = json.JSONDecoder()
    for candidate in candidates:
        for match in re.finditer(r"\{", candidate):
            try:
                payload, _ = decoder.raw_decode(candidate[match.start():].strip())
                if isinstance(payload, dict):
                    return payload
            except json.JSONDecodeError:
                continue
    raise ValueError("Could not extract JSON from model output.")

def parse_model_prediction(raw_text: str):
    try:
        payload = extract_json_object(raw_text)
        scores = {
            "informativeness": clamp_score(payload.get("informativeness", 1)),
            "expressiveness": clamp_score(payload.get("expressiveness", 1)),
            "phatic": clamp_score(payload.get("phatic", 1)),
        }
        ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)
        top_score = ranked[0][1]
        top_labels = [k for k, v in ranked if v == top_score]
        dominant_dimension = normalize_dimension_name(payload.get("dominant_dimension")) or ("mixed" if len(top_labels) > 1 else top_labels[0])
        try:
            confidence = round(float(payload.get("confidence", 0.5)), 4)
        except Exception:
            confidence = 0.5
        confidence = max(0.0, min(1.0, confidence))
        reason = " ".join(str(payload.get("reason", "")).split()).strip() or "strict_json"
        return {
            "informativeness": scores["informativeness"],
            "expressiveness": scores["expressiveness"],
            "phatic": scores["phatic"],
            "dominant_dimension": dominant_dimension,
            "dominant_dimension_score": top_score,
            "confidence": confidence,
            "reason": reason,
        }, True, ""
    except Exception as exc:
        return {
            "informativeness": 1,
            "expressiveness": 1,
            "phatic": 1,
            "dominant_dimension": "mixed",
            "dominant_dimension_score": 1,
            "confidence": 0.0,
            "reason": "parse_fallback",
        }, False, str(exc)

sampling_params = SamplingParams(
    temperature=TEMPERATURE,
    max_tokens=MAX_NEW_TOKENS,
)


## 7) Load dataset

In [ ]:
df = pd.read_csv(DATASET_PATH)
required_cols = ["row_id", "year", "Marque", "Electric", "Hybrid", "Script", "Titre", "Visuel"]
required_cols = [c for c in required_cols if c in df.columns]
sample_df = df.loc[:, required_cols].copy()
for col in [c for c in TEXT_COLUMNS if c in sample_df.columns]:
    sample_df[col] = sample_df[col].map(clean_text)
sample_df["model_input_text"] = sample_df.apply(build_input_text, axis=1)
sample_df = sample_df[sample_df["model_input_text"].str.strip() != ""].copy()
if FULL_RUN_N is not None:
    sample_df = sample_df.head(FULL_RUN_N).copy()
html_box("Columns used", ", ".join(TEXT_COLUMNS))
print("Rows available:", len(sample_df))


## 8) GPU inspection

In [ ]:
def inspect_gpu_state():
    try:
        import torch
        rows = []
        if torch.cuda.is_available():
            for i in range(torch.cuda.device_count()):
                free_bytes, total_bytes = torch.cuda.mem_get_info(i)
                used_gb = (total_bytes - free_bytes) / (1024**3)
                total_gb = total_bytes / (1024**3)
                rows.append({
                    "gpu_index": i,
                    "name": torch.cuda.get_device_name(i),
                    "used_gb": round(used_gb, 2),
                    "free_gb": round(free_bytes / (1024**3), 2),
                    "total_gb": round(total_gb, 2),
                })
        display(pd.DataFrame(rows))
    except Exception as exc:
        print("Torch GPU inspection failed:", exc)


## 9) Chunked inference

In [ ]:
def batched(seq, size):
    for i in range(0, len(seq), size):
        yield seq[i:i+size]

def run_vllm_batch(sample: pd.DataFrame, show_prompt_preview: bool = True):
    llm = None
    rows = []
    try:
        llm = LLM(model=MODEL_NAME, **MODEL_KWARGS)
        inspect_gpu_state()
        prompts = [build_prompt(v, llm=llm) for v in sample["model_input_text"].tolist()]
        row_records = sample.to_dict(orient="records")
        if show_prompt_preview and prompts:
            print(prompts[0][:700])

        start = time.perf_counter()
        total_batches = math.ceil(len(prompts) / PROMPT_BATCH_SIZE)
        for batch_id, (prompt_batch, row_batch) in enumerate(zip(batched(prompts, PROMPT_BATCH_SIZE), batched(row_records, PROMPT_BATCH_SIZE)), start=1):
            print(f"Batch {batch_id}/{total_batches} | prompts={len(prompt_batch)}")
            try:
                outs = llm.generate(prompt_batch, sampling_params, use_tqdm=False)
            except TypeError:
                outs = llm.generate(prompt_batch, sampling_params)
            for in_row, out in zip(row_batch, outs):
                raw_text = out.outputs[0].text if out.outputs else ""
                prediction, parse_ok, parse_error = parse_model_prediction(raw_text)
                rows.append({
                    "row_id": int(in_row["row_id"]),
                    "model_input_text": in_row["model_input_text"],
                    "model_name": MODEL_NAME,
                    "raw_output": raw_text,
                    "prediction": prediction,
                    "parse_ok": parse_ok,
                    "parse_error": parse_error,
                })
        elapsed = max(0.0, time.perf_counter() - start)
        rows_per_second = len(sample) / elapsed if elapsed > 0 else float("inf")
        return rows, elapsed, rows_per_second
    finally:
        try:
            if llm is not None:
                del llm
            gc.collect()
            import torch
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
                torch.cuda.ipc_collect()
        except Exception:
            pass

def flatten_results(all_results: list[dict]) -> tuple[pd.DataFrame, pd.DataFrame]:
    results_df = pd.DataFrame(all_results)
    scores_df = pd.DataFrame([
        {
            "row_id": item.get("row_id"),
            "model_input_text": item.get("model_input_text", ""),
            "informativeness": (item.get("prediction") or {}).get("informativeness"),
            "expressiveness": (item.get("prediction") or {}).get("expressiveness"),
            "phatic": (item.get("prediction") or {}).get("phatic"),
            "dominant_dimension": (item.get("prediction") or {}).get("dominant_dimension"),
            "dominant_dimension_score": (item.get("prediction") or {}).get("dominant_dimension_score"),
            "confidence": (item.get("prediction") or {}).get("confidence"),
            "reason": (item.get("prediction") or {}).get("reason"),
            "model_name": item.get("model_name", ""),
            "parse_ok": item.get("parse_ok", False),
        }
        for item in all_results
    ])
    return results_df, scores_df


## 10) Smoke test

In [ ]:
smoke_df = sample_df.head(SMOKE_TEST_N).copy()
smoke_results, smoke_seconds, smoke_rps = run_vllm_batch(smoke_df, show_prompt_preview=True)
smoke_results_df, smoke_scores_df = flatten_results(smoke_results)
html_box("Smoke test", f"Rows: {len(smoke_scores_df)}\nRows/s: {smoke_rps:.2f}\nParse OK rate: {smoke_scores_df['parse_ok'].mean()*100:.2f}%")


## 11) Full run

In [ ]:
all_results, perf_total_seconds, perf_items_per_second = run_vllm_batch(sample_df, show_prompt_preview=False)
results_df, scores_df = flatten_results(all_results)
n = len(scores_df)
parse_ok_rate = scores_df["parse_ok"].mean() * 100 if n else 0.0
html_box("Full run performance", f"Rows: {n}\nSeconds: {perf_total_seconds:.2f}\nRows/s: {perf_items_per_second:.2f}\nParse OK rate: {parse_ok_rate:.2f}%")


## 12) Save outputs

In [ ]:
jsonl_path = OUTPUT_ROOT / f"{MODEL_SLUG}__predictions_full_{n}.jsonl"
csv_full_path = OUTPUT_ROOT / f"{MODEL_SLUG}__predictions_full_{n}_full.csv"
csv_scores_path = OUTPUT_ROOT / f"{MODEL_SLUG}__predictions_full_{n}_scores.csv"

with open(jsonl_path, "w", encoding="utf-8") as f:
    for item in all_results:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

flat_df = results_df.copy()
flat_df["prediction"] = flat_df["prediction"].apply(lambda x: json.dumps(x, ensure_ascii=False))
flat_df.to_csv(csv_full_path, index=False)
scores_df.to_csv(csv_scores_path, index=False)
html_box("Saved outputs", f"Scores CSV: {csv_scores_path}")


## 13) Sanity check

In [ ]:
html_box(
    "Sanity check",
    f"Rows annotated: {len(scores_df)}\nUnique row_id: {scores_df['row_id'].nunique()}\nMissing score rows: {int(scores_df[['informativeness','expressiveness','phatic']].isna().any(axis=1).sum())}"
)
